# 🤖 LangChain + Bigdata MCP Integration

This notebook demonstrates how your AI agents can interact with **Bigdata.com via MCP (Model Context Protocol)**—a standardized way to connect AI applications to data sources and tools with automatic discovery.

## What This Demonstrates

This notebook demonstrate a cited, multi-source answer in one flow—combining internal portfolios and research with live market data, tearsheets, and calendars. **One MCP connection to Bigdata.com keeps your agents up to date with every new Bigdata.com capability without changing code.** The result: faster, traceable insights with automatic tool discovery and inline source links.

**Bigdata.com MCP Integration:**
- **Automatic Tool Discovery** → MCP exposes all available tools dynamically; when Bigdata.com adds new capabilities (tearsheets, calendars, screeners), your agent gets them automatically
- **Company Lookup** → Resolve tickers to entity IDs via Knowledge Graph
- **Search** → Query news, filings, transcripts with filters
- **Tearsheets** → Company and country financial profiles
- **Events Calendar** → Earnings dates, conference calls

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine MCP-discovered tools with your internal tools seamlessly

**Framework Flexibility:**
> This demo uses **LangChain** with **langchain-mcp-adapters** and **LangSmith** for observability. The MCP protocol is framework-agnostic—**CrewAI**, **AutoGen**, **Google A2A**, and other frameworks can connect to MCP servers using their respective adapters. The key benefit: one integration, automatic access to all current and future Bigdata.com tools.

---

## What is MCP (Model Context Protocol)?

MCP is an open protocol that standardizes how AI applications connect to data sources and tools. Think of it as "USB for AI" - a universal connector.

**Key Benefits:**
- **Automatic Tool Discovery**: MCP servers expose tools dynamically - no manual updates needed when new tools are added
- **Standardized Interface**: One protocol works across all MCP-compatible tools
- **Stateful Connections**: Efficient communication with long-lived sessions

## Architecture

![Agent to Bigdata MCP](./static/agent-mcp.png)


---

## 1️⃣ Install Dependencies

Install from the project root before running this notebook:

```bash
uv sync
```

In [1]:
# Dependencies: install from project root with uv sync (see README)

## 2️⃣ Import Libraries

**LangChain**: ReAct agent and tool-calling

**LangChain MCP Adapters**: Bridge between LangChain and MCP servers

**langgraph_core** (reusable): Environment, `create_financial_database`, `create_vector_store`, local tools (`get_database_tools`, `get_vectorstore_tools`), and display helpers (`display_query`, `display_response`, `display_tools_used`, `display_citations`)

In [2]:
import os
import json
import sqlite3
import random
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML
import html as html_lib

# LangChain
from langchain.tools import tool
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# MCP integration
from langchain_mcp_adapters.client import MultiServerMCPClient

# Reusable core (langgraph_core): environment, data sources, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [3]:
# Environment, database, vector store, and display helpers are provided by langgraph_core
# (see langgraph_core.py). No local definitions needed for reusability.
print("✅ Using langgraph_core for environment, data sources, and display helpers")


✅ Using langgraph_core for environment, data sources, and display helpers


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [4]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project="langgraph-bigdata-mcp-demo",
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n✅ Local data sources ready")

⚠️ LangSmith API key not set. Tracing disabled.
   Set via: export LANGSMITH_API_KEY='your-key'
✅ Bigdata API Key: bd_v2_pzf3...
✅ OpenAI API Key: sk-proj-rC...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

✅ Local data sources ready


## 4️⃣ Load Local Tools

Load tools that interact with local data sources:

In [5]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for t in local_db_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for t in local_vector_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transactions database.

       ...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database including holdings ...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity.

        This sear...


## 5️⃣ Connect to Bigdata MCP Server

**MCP Configuration:**
- **URL**: `https://mcp.bigdata.com/`
- **Transport**: HTTP (streamable)
- **Authentication**: `x-api-key` header

The MCP client automatically discovers all available tools from the server.

In [6]:
# API keys
BIGDATA_API_KEY = os.getenv("BIGDATA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not BIGDATA_API_KEY:
    raise ValueError("BIGDATA_API_KEY not found. Set via: export BIGDATA_API_KEY='your-key'")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Set via: export OPENAI_API_KEY='your-key'")

print(f"✅ Bigdata API Key: {BIGDATA_API_KEY[:10]}...")
print(f"✅ OpenAI API Key: {OPENAI_API_KEY[:10]}...")

✅ Bigdata API Key: bd_v2_pzf3...
✅ OpenAI API Key: sk-proj-rC...


### Configure MCP Client

**Important**: Set `ANYIO_BACKEND='asyncio'` to ensure async backend detection in Jupyter:

In [7]:
# Set async backend for anyio (required in Jupyter)
os.environ['ANYIO_BACKEND'] = 'asyncio'

# Configure MCP client
mcp_client = MultiServerMCPClient(
    {
        "bigdata": {
            "url": "https://mcp.bigdata.com/",
            "transport": "http",  # Streamable HTTP transport
            "headers": {
                "x-api-key": BIGDATA_API_KEY
            }
        }
    }
)

print("✅ MCP client configured")

✅ MCP client configured


### Load MCP Tools

The MCP client automatically discovers all tools exposed by the Bigdata MCP server:

**Note**: In Jupyter notebooks, we use `await` directly instead of `asyncio.run()` because Jupyter already runs an event loop in the background.

In [8]:
# Load tools from Bigdata MCP server
# Note: In Jupyter notebooks, there's already a running event loop, so we use 'await' directly
print("Connecting to Bigdata MCP...")
bigdata_mcp_tools = await mcp_client.get_tools()
print(f"✅ Loaded {len(bigdata_mcp_tools)} tools from Bigdata MCP:")
for tool in bigdata_mcp_tools:
    print(f"   - {tool.name}: {tool.description[:80]}...")

Connecting to Bigdata MCP...
✅ Loaded 5 tools from Bigdata MCP:
   - bigdata_search: Search engine for financial documents, earnings call transcripts, news articles,...
   - bigdata_company_tearsheet: Returns a comprehensive company tearsheet with financial data, market intelligen...
   - bigdata_events_calendar: Returns a professionally formatted markdown calendar of corporate events includi...
   - bigdata_country_tearsheet: Returns a comprehensive country economic tearsheet with economic calendar data.
...
   - find_companies: REQUIRED FIRST STEP: Run this tool whenever the user mentions a company for the ...


## 6️⃣ Combine All Tools

Merge tools from all sources:

In [9]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_mcp_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata MCP tools: {len(bigdata_mcp_tools)}")


✅ Total tools available: 8
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata MCP tools: 5


## 7️⃣ Create LangChain Agent

**LangChain ReAct Agent:**
- **Reasoning**: Plans which tools to use based on user query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

Uses `langchain.agents.create_agent` (the current, non-deprecated API) which creates an agent with:
- Agent executor (LLM with tool calling capabilities)
- Tool registry (all available tools)
- System prompt (guides agent behavior)
- Streaming support (for real-time responses)

In [10]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com MCP):**
- Tools dynamically loaded from Bigdata MCP server
- Tools include news, prices, tear sheet, search, company lookup, and other capital markets capabilities

**Internal Data (Company Systems):**
- `internal_query_database` - Execute SQL queries on portfolio/transaction database
- `internal_portfolio_summary` - Get portfolio holdings and performance summary
- `internal_search_research` - Search internal investment research documents

Guidelines:
- Use appropriate tools based on the query
- For portfolio questions, use internal database tools
- For market intelligence, use Bigdata MCP tools
- Combine multiple sources for comprehensive analysis

**Citation format:** Use inline citations with the **source name as the link text** (not the raw URL). Format as markdown: [Source Name](url) or [1](url), [2](url) so the reader sees a clickable source name. Do not paste full URLs in the body.

**Do not add a separate "Sources" or "References" or "External sources" block at the end** when you have already used inline citations in the text. Inline citations are sufficient.
**Do not offer suggestions for follow up questions**

Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

model = "gpt-5"
# Initialize LLM
llm = ChatOpenAI(
    model=model,
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Create agent using langchain.agents.create_agent (non-deprecated)
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: {model}")
print(f"   System prompt configured")

✅ Agent created with 8 tools
   Model: gpt-5
   System prompt configured


## 8️⃣ Run Example Queries

Let's test the agent with queries that utilize different data sources:

### Example 1 : Multinode

In [11]:
query = """
Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. For each holding, get us the pricing information from tearsheet
4. For each holding, get us negative news
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a consolidated view of PF002 (AI & Semiconductor Focus).

1) Current holdings and performance (internal)
- NVIDIA (NVDA)
  - Shares: 12,000
  - Avg cost: 450.00
  - Current price (internal): 875.50
  - Market value: 10,506,000
  - Unrealized P/L: +5,106,000 (+94.6%)
  - Portfolio weight: 65.7%

- Broadcom (AVGO)
  - Shares: 1,500
  - Avg cost: 850.00
  - Current price (internal): 1,425.00
  - Market value: 2,137,500
  - Unrealized P/L: +862,500 (+67.6%)
  - Portfolio weight: 13.4%

- Palantir (PLTR)
  - Shares: 25,000
  - Avg cost: 18.50
  - Current price (internal): 65.25
  - Market value: 1,631,250
  - Unrealized P/L: +1,168,750 (+252.7%)
  - Portfolio weight: 10.2%

- Advanced Micro Devices (AMD)
  - Shares: 8,000
  - Avg cost: 95.00
  - Current price (internal): 145.25
  - Market value: 1,162,000
  - Unrealized P/L: +402,000 (+52.9%)
  - Portfolio weight: 7.3%

- Taiwan Semiconductor (TSM)
  - Shares: 3,000
  - Avg cost: 110.00
  - Current price (internal): 185.75
  - Market value: 557,250
  - Unrealized P/L: +227,250 (+68.9%)
  - Portfolio weight: 3.5%

Portfolio totals
- Total market value: 15,994,000
- Total unrealized P/L: +7,766,500
- Holdings: 5
- Note: The portfolio is highly concentrated in NVDA (≈66%), which materially drives performance and risk.
(Source: Internal portfolio database PF002)

2) Key risks from internal research (summary)
- Valuation and “AI premium” risk: Sector leaders trading at elevated multiples; risk of multiple compression if AI monetization/distribution of ROI disappoints (Tech Sector Risk Assessment – Jan 2025).
- China exposure and export controls:
  - NVDA: 20–25% of revenue flagged as at-risk from tightening export controls; renewed uncertainty on China-eligible SKUs (Tech Sector Risk Assessment – Jan 2025; NVIDIA thesis update – Dec 2024).
  - TSM: Structural geopolitical exposure (Taiwan), potential fab disruption risks.
- Supply constraints and capex cycles:
  - NVDA/TSM: Supply chain and capacity timing risks; any mis-timed capex cycle (TSMC) can impact utilization and margins.
- Competitive dynamics:
  - AMD: ROCm/software ecosystem lagging CUDA; execution risk on AI accelerators despite strong product cadence (AMD investment thesis – Dec 2024).
  - AVGO: Custom AI silicon opportunities vs. customer concentration across hyperscalers; VMware integration and customer churn risk.
- Policy/regulatory risk: Ongoing tech regulation (US/EU/China), antitrust and platform rules that could affect demand pathways across the stack (Tech Sector Risk Assessment – Jan 2025).
- Concentration risk within PF002: NVDA position size magnifies idiosyncratic outcomes (supply, export, competitive or customer-related shocks).

3) Pricing information from tearsheet (as of latest tearsheets)
- NVIDIA (NVDA) — as of Feb 03, 2026 07:46 PM UTC
  - Last price: $177.19; 1D: -4.54%
  - 52-week range: $86.62 – $212.19
  - Market cap: $4.31T
  - Next earnings: Feb 25, 2026
  - Note: External tearsheet pricing may differ from internal marks due to feed timing and corporate actions.

- Broadcom (AVGO) — as of Feb 03, 2026 07:46 PM UTC
  - Last price: $311.32; 1D: -5.98%
  - 52-week range: $138.10 – $414.61
  - Market cap: $1.48T
  - Next earnings: Mar 4, 2026

- Palantir (PLTR) — as of Feb 03, 2026 07:45 PM UTC
  - Last price: $156.41; 1D: +5.84%
  - 52-week range: $66.12 – $207.52
  - Market cap: $357.29B
  - Next earnings: May 4, 2026

- Advanced Micro Devices (AMD) — as of Feb 03, 2026 07:46 PM UTC
  - Last price: $240.86; 1D: -2.20%
  - 52-week range: $76.48 – $267.08
  - Market cap: $392.13B
  - Next earnings: Feb 3, 2026

- Taiwan Semiconductor (TSM) — as of Feb 03, 2026 05:35 AM UTC (TAI; TWD)
  - Last price: NT$1,800; 1D: +1.98%
  - 52-week range: NT$780 – NT$1,830
  - Market cap: NT$46.67T
  - Next earnings: Apr 16, 2026

4) Negative news (recent highlights, last ~2 weeks)
- NVIDIA (NVDA)
  - OpenAI investment uncertainty; CEO Jensen Huang downplays a reported “up to $100B” plan; headlines indicate talks have slowed and OpenAI evaluating alternatives [Benzinga - Feb 02, 2026](https://www.benzinga.com/node/50304821?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [Business Insider - Feb 03, 2026](https://markets.businessinsider.com/news/stocks/nvidia-stock-falls-openai-says-best-ai-chips-gigantic-customer-but-eyes-alternatives-1035777865).
  - Reports of H200 component production pauses tied to China customs actions create supply/revenue uncertainty [Financial Times - Jan 22, 2026](https://www.ft.com/content/ee710a36-00c4-4635-9c14-5d438e4e2281), [Yahoo! Finance - Jan 21, 2026](https://finance.yahoo.com/news/h200-component-production-halted-means-170337192.html).
  - Sentiment/positioning pressure amid options and multiple compression concerns [Benzinga - Jan 23, 2026](https://www.benzinga.com/node/50110888?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

- Broadcom (AVGO)
  - VMware channel disruption and partner backlash in Europe as programs get pared back; potential customer transitions and pricing pushback [The Register - Jan 31, 2026](https://www.theregister.com/2026/01/31/broadcom_vmware_cloud_partners/), [MSN - Jan 31, 2026](https://www.msn.com/en-us/money/news/broadcom-bulldozes-vmware-cloud-partners-as-march-deadline-looms/ar-AA1Vnwph?ocid=finance-verthp-feeds).
  - Legal friction with major client over software access (settled, but highlights integration tension post-VMware) [Yahoo! Finance - Feb 02, 2026](https://finance.yahoo.com/news/fidelity-resolves-legal-dispute-around-141117984.html).
  - Margin sensitivity: mix shift toward lower-margin AI silicon could pressure gross margins vs. software [Yahoo! Finance - Feb 02, 2026](https://finance.yahoo.com/news/broadcom-shares-slide-avgo-stock-175613449.html).

- Palantir (PLTR)
  - RBC warns of potential 70% downside; slower gov’t contract trackers and commercial durability questioned [CNBC - Jan 27, 2026](https://www.cnbc.com/2026/01/27/rbc-sees-a-bunch-of-red-flags-on-palantir-ahead-of-earnings.html), [Benzinga - Feb 02, 2026](https://www.benzinga.com/node/50301699?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Reputational/governance risk from ICE surveillance contracts; valuation scrutiny persists [U.S. News & World Report - Feb 02, 2026](https://money.usnews.com/investing/news/articles/2026-02-02/palantir-ceo-defends-surveillance-tech-as-us-government-contracts-boost-sales), [Nasdaq - Jan 29, 2026](https://www.nasdaq.com/articles/why-palantir-technologies-stock-slumped-today), [Financial Times - Feb 02, 2026](https://www.ft.com/content/99fdde76-a545-4a31-b1bb-0034a4e902f0).

- Advanced Micro Devices (AMD)
  - Reported hiccup/delay for MI450 AI GPU development, which could affect competitive cadence vs. NVDA [Nasdaq - Jan 30, 2026](https://www.nasdaq.com/articles/why-advanced-micro-devices-stock-just-dropped).
  - Broader risk context (execution on rack-scale, demand lumpiness, reliance on single foundry, console cycle maturity) discussed in recent coverage [Miami Herald (syndicated) - Jan 20, 2026](https://www.miamiherald.com/news/business/article314391070.html#storylink=partnerdigest_the).

- Taiwan Semiconductor (TSM)
  - Strategic/geopolitical risk: dependence on Taiwan; debate that taking TSMC “off the board” is a strategic lever in great-power competition [Stratechery - Jan 26, 2026](https://stratechery.com/2026/tsmc-risk/).
  - Capex timing risk: CEO cautions on “nervous” AI demand forecasting and risk of overbuild amid NT$1.7T+ valuation and aggressive 2026 capex plan [Yahoo! Finance - Jan 20, 2026](https://finance.yahoo.com/news/m-very-nervous-tsmc-ceo-194918758.html).
  - Policy/market pressure headlines around overseas expansion and local sell-offs tied to macro/politics [Nikkei Asia - Feb 03, 2026](https://asia.nikkei.com/opinion/tsmc-s-american-expansion-is-not-a-surrender-it-s-insurance-for-taiwan), [Focus Taiwan - Jan 21, 2026](https://focustaiwan.tw/business/202601210015).

Notes
- Internal performance figures reflect our internal marks/feeds; tearsheet prices are real-time public market snapshots and may differ due to timing, FX, listings, and corporate actions.
- Portfolio concentration in NVDA is the primary driver of both upside and downside. Internal research continues to flag valuation sensitivity, China/export-control exposure, and capex/supply-timing risks across the stack.

### Example 2: Bigdata MCP Tool Query

Use external market intelligence:

In [12]:
# Example 2: Use Bigdata MCP tools for external data
query = "Find the latest news about NVIDIA's earnings and revenue growth using Bigdata tools."

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
#display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

In [13]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

Here’s the latest coverage on NVIDIA’s earnings and revenue growth from the past few days:

- Multiple outlets continue to highlight NVIDIA’s most recent reported quarter (Q3 FY2026): revenue of about $57.0B (+62% YoY), with record Data Center revenue of ~$51.2B (+66% YoY) and GAAP gross margin ~73.4%. Management guided Q4 revenue to ~$65B, underscoring continued momentum in AI infrastructure demand [Yahoo! Finance - Feb 3, 2026](https://finance.yahoo.com/news/nvidia-may-not-invest-100-162024096.html), [Business Insider - Feb 3, 2026](https://markets.businessinsider.com/news/stocks/nvidia-stock-forecast-bracing-for-a-pullback-amid-ai-headwinds-1035775356), [Benzinga - Nov 19, 2025](https://www.benzinga.com/node/48962938?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

- Recent commentary reiterates that the latest results beat expectations and that the near-term outlook remains strong, driven by data center GPUs, networking (NVLink, InfiniBand, Spectrum-X), and ongoing AI buildouts. Articles emphasize that NVIDIA is still “firmly in control of the AI race” following the record Q3 and stronger-than-expected Q4 guidance [Yahoo! Finance - Feb 3, 2026](https://finance.yahoo.com/news/nvidia-may-not-invest-100-162024096.html), [Business Insider - Feb 3, 2026](https://markets.businessinsider.com/news/stocks/nvidia-stock-forecast-bracing-for-a-pullback-amid-ai-headwinds-1035775356).

- A fresh angle in the news flow: potential initial H200 shipments to China are being discussed as a possible incremental tailwind, with coverage still tying back to NVIDIA’s strong Q3 revenue growth and margins [Yahoo! Finance - Feb 2, 2026](https://finance.yahoo.com/news/first-batch-h200-shipments-could-151100716.html).

- Peer-comparison write-ups this week also spotlight NVIDIA’s outsized revenue growth versus the semiconductor industry. One summary pegs NVIDIA’s revenue growth at ~62.5%, handily ahead of an industry average near ~38% in the sample set reviewed [Benzinga - Feb 3, 2026](https://www.benzinga.com/node/50335828?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [Benzinga - Feb 2, 2026](https://www.benzinga.com/node/50301233?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

- For the underlying numbers, the company’s Q3 FY2026 release remains the anchor for current coverage: revenue $57.0B (+62% YoY), Data Center $51.2B (+66% YoY), gross margin ~73.4%, and Q4 revenue outlook ~$65B [Benzinga (Press release relay) - Nov 19, 2025](https://www.benzinga.com/node/48962938?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### Example 3: Multi-Source Query

Combine local and external data:

In [14]:
# Example 3: Multi-source comprehensive analysis
query = """For our NVIDIA holdings:
1. Check our internal database to see which portfolios hold NVDA and how much
2. Search our internal research for our investment thesis
3. Use Bigdata tools to find recent news about NVIDIA
4. Provide a comprehensive summary combining all sources"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a consolidated view of our NVIDIA position, thesis, and the latest developments.

1) Internal holdings (current internal DB snapshot)
- PF002 – AI & Semiconductor Focus
  - Shares: 12,000
  - Avg cost: $450.00
  - Current price: $875.50
  - Market value: $10,506,000
  - Unrealized P&L: $5,106,000
  - Portfolio AUM: $15,000,000
  - Weight: 70.0%

- PF003 – Diversified Tech Leaders
  - Shares: 8,000
  - Avg cost: $520.00
  - Current price: $875.50
  - Market value: $7,004,000
  - Unrealized P&L: $2,844,000
  - Portfolio AUM: $50,000,000
  - Weight: 14.0%

- Combined across PF002 and PF003
  - Total shares: 20,000
  - Total market value: $17,510,000
  - Total unrealized P&L: $7,950,000
  - Blended cost basis: ~$478/share

2) Internal research — our investment thesis (highlights)
- Core thesis (Investment Thesis Update, 2024-12-15)
  - Data center GPU leadership: explosive demand for H100/H200 accelerators drove outsized data center revenue growth; next-gen Blackwell architecture (B100/B200) expected to extend performance lead.
  - Durable software moat: CUDA ecosystem (millions of developers) creates switching costs and reinforces platform lock-in.
  - Inference runway: expanding enterprise AI deployment supports large and growing inference TAM over the next several years.
  - Valuation framework (then-current): STRONG BUY, PT $950 on ~25x FY26E EPS.
- Portfolio strategy (Q1 2025)
  - Recommended increasing NVDA weight (+3%) based on sustained AI training demand and pipeline visibility.
- Key risks (Risk Assessment, Jan 2025)
  - China export controls: 20–25% of revenue at risk from tightening restrictions.
  - Competitive pressure: AMD’s MI300X improving, but software ecosystem (ROCm) still trails CUDA.
  - Supply constraints and potential AI-spend normalization/ROI uncertainty could pressure multiples.

3) Recent NVIDIA news (last 7 days)
- Dassault Systèmes partnership: NVIDIA and Dassault announced a long-term strategic partnership to build a shared industrial AI architecture integrating Dassault’s Virtual Twin with NVIDIA AI infrastructure and software libraries, aiming to enable “industry World Models” and agentic workflows on 3DEXPERIENCE [The Fly](https://app.bigdata.com/documents/878BDF26ED0ABAB691AA3AC6056E7DD9).
- Cyngn collaboration: Cyngn advanced its work with NVIDIA via a simulation environment built on NVIDIA Isaac Sim to accelerate deployment of autonomous vehicle solutions in warehouse/industrial settings [Benzinga](https://www.benzinga.com/node/50326650?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
- Robotaxi ecosystem push: Mercedes-Benz, with NVIDIA and Uber, is progressing plans to launch robotaxi services using S-Class vehicles across major markets, emphasizing safety redundancies and MB.OS integration [MT Newswires](https://app.bigdata.com/documents/FAC2144CEF43B58817D782BD1CB7F258).
- OpenAI relationship/rumors: Reports of a stalled NVIDIA-OpenAI investment weighed on shares; OpenAI’s Sam Altman later reaffirmed NVIDIA as the “best AI chips” provider and a long-term key supplier, helping stabilize sentiment [Nasdaq](https://www.nasdaq.com/articles/us-stocks-may-see-early-weakness-nvidia-slumps), [Benzinga](https://www.benzinga.com/node/50323803?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
- China chip sales context: Reuters-sourced report indicated Alibaba, ByteDance, and Tencent received approval to purchase >400,000 NVIDIA H200 AI chips, signaling ongoing demand under the current regulatory regime [MT Newswires](https://app.bigdata.com/documents/47B7737C06B514353C732D96B353C8EE).

4) Combined view and implications for our holdings
- Thesis alignment: The Dassault and Cyngn updates reinforce NVIDIA’s ecosystem and software-first advantage in industrial and “physical AI,” consistent with our CUDA moat and platform thesis. The robotaxi collaboration with Mercedes-Benz underscores automotive/AV compute optionality.
- Demand signals vs. noise: Despite short-term volatility tied to headlines around OpenAI and capex scrutiny in mega-cap tech, subsequent management and partner communications point to intact strategic relationships and continued multi-tenant demand for advanced accelerators.
- China/regulatory backdrop: The reported approvals for H200 shipments are supportive near term, but our risk framework remains unchanged—export policy remains a meaningful variable. Portfolio risk monitoring should keep an eye on any incremental U.S.-China restrictions that could impact mix and visibility.
- Portfolio positioning: NVDA is a high-conviction, high-concentration position in PF002 (70% weight) and a significant holding in PF003 (14%). Performance contribution is strong (combined unrealized P&L ~$7.95M), but concentration risk in PF002 is elevated relative to typical diversification guardrails. Align ongoing risk oversight (exposure limits, drawdown controls, and hedge strategies) with our earlier risk assessments while maintaining the core long-term thesis.

If you’d like, I can refresh this with an events/earnings schedule or pull a tearsheet for latest financials, estimates, and analyst revisions.

### Example 4: Company Briefing 

In [15]:
query = """
Summarize recent developments for CoreWeave (last 30 days).
**Steps:**
1. Call find_companies and get the company id
2. Call bigdata_tearsheet and get business context
3. Use bigdata_search and find news in the last 30 days
4. Categorize findings
**Categories:**
- Financial results
- Product/tech launches
- M&A and partnerships
- Regulatory/legal updates
- Management changes
- Other material events
For each: Date, facts, investment implications (bullish/bearish/neutral).
Please add inline source attribution.
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)



In [16]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

CoreWeave — recent developments (last 30 days)

Business context
- CoreWeave provides GPU-accelerated cloud infrastructure for AI workloads and positions itself as “The Essential Cloud for AI,” serving AI training/inference, VFX, and enterprise AI at scale [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm).

Financial results
- None reported in the last 30 days. No new quarterly earnings releases were issued during this window.

Product/tech launches
- Jan 26, 2026 — NVIDIA/CoreWeave technical expansion: CoreWeave to adopt NVIDIA’s CPU and storage platforms and support multiple NVIDIA generations; plan to offer CoreWeave software to global CSPs and enterprises as part of a deeper alignment to support AI workloads at scale. Facts: Part of a broader initiative to accelerate buildout of “AI factories” and expand software/platform offerings [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm), [Benzinga](https://www.benzinga.com/node/50123767?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Investment implications: Bullish. Enhances CoreWeave’s tech stack breadth and partner distribution, supports capacity scaling for GenAI demand, and opens a higher-margin software/platform pathway.

M&A and partnerships
- Jan 26, 2026 — Expanded NVIDIA partnership + $2B strategic equity investment: CoreWeave and NVIDIA announced an expanded relationship to accelerate the buildout of more than 5GW of AI factories by 2030. NVIDIA invested $2B at $87.20/share and will support CoreWeave in securing land/power/shell capacity and go-to-market for CoreWeave software [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm), [The Fly](https://app.bigdata.com/documents/8A18A14BF303B27A908A3D7C53C9847D), [Benzinga](https://www.benzinga.com/node/374BC7CCFFFFAABF89E8DBAF1EE5A5FB?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Investment implications: Bullish. Capital infusion mitigates funding risk for capacity buildout; NVIDIA’s operational and distribution support reduces execution risk and may accelerate revenue scaling.

Regulatory/legal updates
- Jan 28 – Feb 3, 2026 — Multiple class-action filings/solicitations: Several shareholder law firms announced class actions or investigations alleging CoreWeave misled investors about scalability, data center execution (including delays at a Denton, TX cluster), and infrastructure risk concentration; lead-plaintiff deadlines cited around Mar 13, 2026 [Hagens Berman via Benzinga](https://www.benzinga.com/node/50331453?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [BFA Law via Benzinga](https://www.benzinga.com/node/50325398?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [Pomerantz via Benzinga](https://www.benzinga.com/node/50231987?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [Robbins LLP via Benzinga](https://www.benzinga.com/node/50318641?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Investment implications: Bearish. Litigation elevates headline and liability risk, could pressure valuation multiples, and may distract management during a critical scale-up period.

Management changes
- None reported in the last 30 days.

Other material events
- Jan 26–27, 2026 — Analyst actions and market reaction:
  - DA Davidson upgraded CoreWeave to Buy (PT to $110 from $68) on the heels of the NVIDIA deal; shares jumped ~16% intraday to ~$107.50 following the investment/upgrade news [The Fly](https://app.bigdata.com/documents/5E96637BA164E8BA3F8B22A3D71F8E38), [The Fly](https://app.bigdata.com/documents/5188A35AE68DD1E5F74D3E11EF506CA2).
  - Deutsche Bank upgraded to Buy with a $140 PT, citing a solid medium-term outlook and potential for higher 2026 revenue estimates if capacity is delivered as planned [The Fly](https://app.bigdata.com/documents/852B56AA84FD6006DD319A8B49928A9C), [MT Newswires](https://app.bigdata.com/documents/188A5FE5C910F1A3858209730BD31EAC).
  - Jefferies highlighted that NVIDIA’s support lowers execution risk and creates a high-margin, asset-light software growth channel; it also noted NVIDIA’s stake rising to about 9% from ~7% as a show of conviction [The Fly](https://app.bigdata.com/documents/B82AC6BB51044D05FBECEFF110A3E2E4).
  - Investment implications: Bullish. Upgrades and positive sell-side commentary reinforce improved risk/reward following NVIDIA’s investment and partnership expansion; constructive for sentiment and cost of capital.

### Example 5: Local Database Query

Query internal portfolio holdings:

In [17]:
# Example 5: Query internal database for top holdings
query = "What are our top 5 holdings by market value across all portfolios?"

# Display query
display_query(query)

# Run agent (async invocation in Jupyter)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)


Top 5 holdings by total market value (all portfolios combined):
1. NVDA — NVIDIA Corporation — $17,510,000
2. MSFT — Microsoft Corporation — $9,556,500
3. AAPL — Apple Inc. — $7,410,000
4. CRM — Salesforce Inc. — $3,255,000
5. META — Meta Platforms Inc. — $2,632,500

### Example 6: Local Vector Store Query

Search internal research documents:

In [18]:
# Example 6: Search internal vector store
query = "What does our internal research say about NVIDIA's competitive moat?"

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)


Here’s how our internal research characterizes NVIDIA’s competitive moat:

- Software ecosystem lock-in (primary moat): CUDA’s scale (4M+ developers) and mature libraries/toolchains create high switching costs and faster time-to-solution versus alternatives. This “developer gravity” is cited as the key differentiator sustaining share and pricing power (NVIDIA Q4 2024 Investment Thesis Update — Dec 15, 2024).

- Full‑stack integration advantage: Tight coupling of GPUs with NVLink/NVSwitch networking, systems (e.g., DGX), and a robust software stack delivers out‑of‑the‑box performance and reliability that competitors struggle to match, reinforcing customer preference and stickiness (NVIDIA Q4 2024 Investment Thesis Update — Dec 15, 2024).

- Performance leadership and cadence: H100/H200 leadership in training workloads and the Blackwell (B100/B200) roadmap with materially higher performance (quoted as ~2.5x) extend the lead and give customers confidence in multi‑year roadmaps, supporting long‑term commitments (NVIDIA Q4 2024 Investment Thesis Update — Dec 15, 2024).

- Mindshare with hyperscalers and enterprises: NVIDIA remains the default platform for state‑of‑the‑art training; as inference scales enterprise‑wide, the installed base and software familiarity are expected to carry over, expanding the moat into inference TAM (NVIDIA Q4 2024 Investment Thesis Update — Dec 15, 2024).

What could erode the moat (risks we flag):
- AMD competition: MI300X shows credible performance (notably high HBM capacity and solid LLM inference), but ROCm’s ecosystem still lags CUDA—our view is that software maturity remains AMD’s main headwind to displacing NVIDIA at scale (AMD – Data Center & AI Opportunity Assessment — Dec 11, 2024).
- Geopolitical/export controls: 20–25% of NVDA revenue exposed to China restrictions; supply chain constraints can limit upside and frustrate customers (Technology Sector Risk Assessment — Jan 10, 2025).
- Macro/AI ROI risk: If infrastructure spend runs ahead of realized ROI, budgets could normalize and compress the AI premium (Technology Sector Risk Assessment — Jan 10, 2025).

Bottom line from our research:
- Moat quality: High and durable over the next 12–24 months, anchored by CUDA/software lock‑in plus full‑stack integration and performance cadence.
- Watch items: ROCm/software progress at AMD, pace of enterprise inference adoption vs. budget discipline, and export‑control impacts.
- Positioning signal: Our strategy team increased NVDA weight (+3%) on conviction that moat advantages will persist as AI training and inference scale (Q1 2025 Portfolio Strategy — Jan 5, 2025).

## 9️⃣ Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**
- Portfolio/holdings questions → `internal_query_database` or `internal_portfolio_summary`
- Internal research → `internal_search_research`
- External market data → Bigdata MCP tools (automatically discovered)

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

## 🔟 Key Benefits of This Architecture

### 1. **Automatic Tool Discovery**
- MCP server exposes tools dynamically
- No code changes needed when Bigdata.com adds new tools
- Agent automatically learns about new capabilities

### 2. **Unified Interface**
- Single agent interface for all data sources
- Consistent tool calling pattern
- Easy to add more MCP servers or local tools

### 3. **Stateful Reasoning**
- LangGraph maintains conversation state
- Agent can reference previous tool results
- Multi-turn reasoning supported

### 4. **Observability**
- LangSmith tracing shows full execution flow
- Easy to debug tool selection and results
- Performance monitoring built-in

## 🎯 Next Steps

**Extend this architecture:**

1. **Add More MCP Servers**
   ```python
   mcp_client = MultiServerMCPClient({
       "bigdata": {...},
       "other_mcp_server": {...}
   })
   ```

2. **Custom Local Tools**
   - Create @tool decorated functions
   - Add to tool list before agent creation

3. **Advanced Graph Patterns, based on need**
   - Use `StateGraph` for custom control flow
   - Add conditional edges for routing logic
   - Implement human-in-the-loop

4. **Persistent Memory**
   - Add checkpointer for conversation history
   - Use `MemorySaver` or Redis for state persistence

5. **Context Compression**
   - Strategy to compress the context (i.e. summarizing when context window reaches 80%)

**References:**
- LangGraph: https://langchain-ai.github.io/langgraph/
- MCP Adapters: https://reference.langchain.com/python/langchain_mcp_adapters/
- Bigdata MCP: https://docs.bigdata.com/mcp-reference/

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **MCP Protocol Spec**: https://modelcontextprotocol.io
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com